# 01 — Data and leakage audit

**Data:** UCI Bank Marketing, `bank-additional-full.csv` (Moro, Cortez & Rita, 2014), licensed CC BY 4.0.
Source: https://archive.ics.uci.edu/dataset/222/bank+marketing

## Scoring decision point

A lead is scored **immediately before the recorded outbound call is dialled**. A column may feed the deployable
model only if its value is known at that moment. Availability is judged on that basis alone, not on whether the
column improves validation performance.

All feature definitions live in `src/features.py`; this notebook only displays them. No model is trained here.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from src import audit, features as ft
from src.data import load_raw

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 100)
raw = load_raw()
raw.shape

(41188, 21)

## 1. Schema and file integrity

In [2]:
audit.check_schema(raw)

{'n_rows': 41188,
 'n_columns': 21,
 'columns_match_expected': True,
 'missing_columns': [],
 'unexpected_columns': [],
 'n_nan_cells': 0,
 'n_exact_duplicate_rows': 12}

The 12 exact duplicate rows are kept. The file has no client identifier, so identical rows cannot be told
apart from two genuinely identical leads, and removing them would not change any conclusion at this scale.

## 2. Row ordering

The file has no date column. The codebook states rows are ordered by date from May 2008 to November 2010.
The check below infers a calendar period from the month sequence (the year advances whenever the month number
drops) and confirms each period forms one contiguous run. The inferred period is used for auditing only and is
**not** a model feature.

In [3]:
audit.check_ordering(raw)

{'n_month_runs': 26,
 'each_period_is_one_contiguous_run': True,
 'first_period': '2008-05',
 'last_period': '2010-11',
 'consistent_with_documented_range': True}

In [4]:
audit.period_summary(raw)

,period,first_row,n_rows,conversion_rate,n_unique_emp.var.rate,n_unique_cons.price.idx,n_unique_cons.conf.idx,n_unique_euribor3m,n_unique_nr.employed
0,2008-05,0,7763,0.030916,1,1,1,7,1
1,2008-06,7763,4374,0.042981,1,1,1,11,1
2,2008-07,12137,6685,0.060883,1,1,1,11,1
3,2008-08,18822,5175,0.052367,1,1,1,8,1
4,2008-10,23997,67,0.626866,1,1,1,11,1
5,2008-11,24064,3616,0.052544,1,1,1,18,1
6,2008-12,27680,10,0.100000,1,1,1,9,1
7,2009-03,27690,282,0.446809,1,1,1,22,1
8,2009-04,27972,2458,0.179821,1,1,1,17,1
9,2009-05,30430,5794,0.090438,1,1,1,14,1


The conversion rate is far from stable over time, and the row volume is concentrated in 2008.
`cons.price.idx`, `cons.conf.idx`, `emp.var.rate` and `nr.employed` take a single value within each month run.

## 3. Class balance

In [5]:
audit.class_balance(raw)

{'n_yes': 4640, 'n_no': 36548, 'conversion_rate': 0.11265417111780131}

Describing the contiguous development / validation / test segments (no modelling, descriptive only).
The test segment is the final 20% of rows, fixed in Phase 1. Validation starts at the month boundary closest to
70% of rows (the start of April 2009), chosen from chronology alone.

In [6]:
audit.contiguous_segment_summary(raw)

,segment,first_row,last_row,n_rows,share_of_all_rows,first_period,last_period,conversion_rate,n_pdays_recorded,share_pdays_recorded,n_previous_gt_0,share_previous_gt_0,n_poutcome_success,share_cellular,months_present
0,development,0,27971,27972,0.679130,2008-05,2009-03,0.052374,39,0.001394,756,0.027027,38,0.509009,may jun jul aug oct nov dec mar
1,validation,27972,32948,4977,0.120836,2009-04,2009-05,0.127587,135,0.027125,1519,0.305204,130,0.936307,apr may
2,test,32949,41187,8239,0.200034,2009-05,2010-11,0.308290,1341,0.162762,3350,0.406603,1205,0.879476,may jun jul aug sep oct nov dec mar apr


The base rate rises sharply across the three segments, and prior-campaign history is rare in the development
segment. A model trained on the early period will be evaluated on a visibly different population.
This is the main limitation of the dataset for a time-ordered evaluation, and it is reported rather than hidden
by shuffling.

## 4. Feature-availability table

In [7]:
ft.availability_table()

,raw_column,availability_at_scoring,decision,model_features,in_primary_model,sensitivity_only,rationale
0,age,available,use,age,True,False,Client attribute held by the bank before the call.
1,job,available,use,job,True,False,Client attribute held before the call; 'unknown' kept as a category.
2,marital,available,use,marital,True,False,Client attribute held before the call; 'unknown' kept as a category.
3,education,available,use,education,True,False,Client attribute held before the call; 'unknown' kept as a category.
4,default,available,use,default,True,False,Bank record held before the call; 'unknown' kept as a category.
5,housing,available,use,housing,True,False,Bank record held before the call; 'unknown' kept as a category.
6,loan,available,use,loan,True,False,Bank record held before the call; 'unknown' kept as a category.
7,contact,available,use,contact,True,False,"Channel of the planned call, chosen before dialling."
8,month,available,sensitivity only,month,False,True,"Month of the call being dialled, known at dial time, so not a leakage risk. Dropped from the primary model on generalisation grounds: with 26 months of data its learned effect is a calendar-period effect, and later months (e.g. March, April, September, December) are absent or rare in earlier training data. Evaluated only in the month sensitivity set."
9,day_of_week,available,use,day_of_week,True,False,"Weekday of the call being dialled, known at dial time."


### Notes on the non-obvious decisions

- **`duration`** is unavailable at the decision point and is excluded from every feature set.
- **`campaign`** includes the contact being scored (observed minimum 1). The deployable feature is
  `prior_contacts_current_campaign = campaign - 1`. A remaining caveat: each row is the *last* contact of the
  campaign for that client, and calling stops once a client subscribes, so the data under-represents earlier
  attempts that did not end the sequence.
- **`pdays`**: 999 is a sentinel, not a number of days. It becomes `prior_contact_days_known` (0/1) and
  `days_since_prior_contact` (missing when the sentinel is present). The column is available, but only 6 of the
  24,712 rows of the original training segment (39 of 27,972 in the revised development segment) have a recorded
  value, so both derived features are kept out of the primary model and used
  only in a labelled sensitivity set. That is a usefulness decision, not an availability one.
- **`month`, `day_of_week`** describe the call being dialled and are available (no leakage). `day_of_week` is in
  the primary model. `month` is dropped from it on generalisation grounds: with 26 months of data its learned effect
  is a calendar-period effect, and later months are absent or rare in earlier training data. It is evaluated only
  in a labelled sensitivity set.
- **Economic indicators** are excluded from the primary model. The file attaches a same-period value to each
  call, and the dataset does not document whether that value was available as of the call time. They are kept
  only in an `extended` set for a labelled sensitivity analysis.

In [8]:
print("Primary (deployable):", ft.feature_names(ft.PRIMARY))
for fs in (ft.PDAYS_SENSITIVITY, ft.EXTENDED):
    print(f"{fs} adds:", [f for f in ft.feature_names(fs) if f not in ft.feature_names(ft.PRIMARY)])
print("Forbidden:", sorted(ft.FORBIDDEN_COLUMNS))

Primary (deployable): ['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'day_of_week', 'prior_contacts_current_campaign', 'previous', 'poutcome']
pdays_sensitivity adds: ['prior_contact_days_known', 'days_since_prior_contact']
extended adds: ['emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']
Forbidden: ['duration', 'y']


## 5. Unknown values and sentinels

In [9]:
pd.Series(audit.unknown_shares(raw), name='share_unknown').sort_values(ascending=False)

default        0.208726
education      0.042027
housing        0.024036
loan           0.024036
job            0.008012
marital        0.001942
contact        0.000000
month          0.000000
day_of_week    0.000000
poutcome       0.000000
Name: share_unknown, dtype: float64

In [10]:
audit.sentinel_checks(raw)

{'campaign_min': 1,
 'campaign_max': 56,
 'share_pdays_sentinel': 0.9632174419733903,
 'pdays_max_excluding_sentinel': 27,
 'n_previous_gt_0_with_pdays_sentinel': 4110,
 'poutcome_nonexistent_iff_previous_0': True,
 'n_duration_0': 4,
 'n_duration_0_converted': 0,
 'n_default_yes': 3}

In [11]:
pd.crosstab(raw['pdays'].ne(ft.PDAYS_SENTINEL).rename('pdays recorded'), raw['poutcome'])

poutcome,failure,nonexistent,success
pdays recorded,,,
False,4110,35563,0
True,142,0,1373


- `unknown` is kept as its own category; nothing is imputed from the data.
- `default` is `unknown` for about a fifth of rows and `yes` for only three, so it carries little beyond
  "known to have no default" versus "not known".
- Some rows have earlier-campaign contacts (`previous > 0`) but the `pdays` sentinel, so the `pdays` indicator
  means "days recorded", not "ever contacted". `previous` and `poutcome` carry the latter.
- `poutcome = nonexistent` coincides exactly with `previous = 0`.

## 6. Built features

In [12]:
X = ft.build_features(raw)  # primary set
y = ft.make_target(raw)
ft.assert_no_leakage(X.columns)
X.head()

,age,job,marital,education,default,housing,loan,contact,day_of_week,prior_contacts_current_campaign,previous,poutcome
0,56,housemaid,married,basic.4y,no,no,no,telephone,mon,0,0,nonexistent
1,57,services,married,high.school,unknown,no,no,telephone,mon,0,0,nonexistent
2,37,services,married,high.school,no,yes,no,telephone,mon,0,0,nonexistent
3,40,admin.,married,basic.6y,no,no,no,telephone,mon,0,0,nonexistent
4,56,services,married,high.school,no,no,yes,telephone,mon,0,0,nonexistent


## Limitations carried forward

1. No timestamps or client identifiers: ordering is inferred from the month sequence, and the same client may
   appear in more than one period.
2. Strong drift in base rate, contact channel and availability of prior-campaign history across the time-ordered
   segments.
3. Rows are final contacts only, selected partly by outcome.
4. One bank, one country, 2008-2010, spanning the financial crisis. Results are a backtest, not a forecast of
   conversion elsewhere.